In [1]:
%pip install openai pandas pydantic

   ---------------------------------------- 0.0/1.7 MB ? eta -:--:--
   ------------------------ --------------- 1.0/1.7 MB 8.5 MB/s eta 0:00:01
   ---------------------------------------- 1.7/1.7 MB 10.2 MB/s eta 0:00:00
  Attempting uninstall: idna
    Found existing installation: idna 3.10
    Uninstalling idna-3.10:
      Successfully uninstalled idna-3.10
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [64]:
import pandas as pd
from dotenv import load_dotenv
from openai import OpenAI
from pydantic import BaseModel, Field

# os.environ["OPENAI_API_KEY"] = "your-key"
load_dotenv(override=True)

client = OpenAI()
print("Environment and OpenAI Client successfully initialized.")

Environment and OpenAI Client successfully initialized.


In [65]:
import mock_services as gc

tools_map = {
    "get_order_details": gc.get_order_details,
    "get_user_profile": gc.get_user_profile,
    "check_return_policy": gc.check_return_policy,
    "process_refund": gc.process_refund,
}

tools_schema = [
    {
        "type": "function",
        "function": {
            "name": "get_order_details",
            "description": gc.get_order_details.__doc__,
            "parameters": {
                "type": "object",
                "properties": {
                    "order_id": {
                        "type": "string",
                        "description": "The exact order identifier, e.g., 'ORD-1001'.",
                    }
                },
                "required": ["order_id"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_user_profile",
            "description": gc.get_user_profile.__doc__,
            "parameters": {
                "type": "object",
                "properties": {
                    "user_id": {
                        "type": "string",
                        "description": "The exact user identifier, e.g., 'USR-101'.",
                    }
                },
                "required": ["user_id"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "check_return_policy",
            "description": gc.check_return_policy.__doc__,
            "parameters": {
                "type": "object",
                "properties": {
                    "order_id": {"type": "string", "description": "The order identifier."},
                    "reason": {
                        "type": "string",
                        "enum": [
                            "damaged_on_arrival",
                            "wrong_item",
                            "item_missing",
                            "late_delivery",
                            "changed_mind",
                        ],
                        "description": "Reason for the return/refund request.",
                    },
                },
                "required": ["order_id"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "process_refund",
            "description": gc.process_refund.__doc__,
            "parameters": {
                "type": "object",
                "properties": {
                    "order_id": {"type": "string", "description": "The order identifier."},
                    "amount": {"type": "number", "description": "The amount to refund in USD."},
                    "reason": {
                        "type": "string",
                        "enum": [
                            "damaged_on_arrival",
                            "wrong_item",
                            "item_missing",
                            "late_delivery",
                            "changed_mind",
                        ],
                        "description": "Reason for the refund.",
                    },
                },
                "required": ["order_id", "amount"],
            },
        },
    },
]

In [67]:
class ActionTaken(BaseModel):
    decision: str = Field(description="Final decision: APPROVED, REJECTED, or ESCALATION_REQUIRED")
    refund_amount: float = Field(description="Approved refund amount in USD. 0.0 if not approved.")
    tools_called: list[str] = Field(
        description="List of distinct tool names executed during resolution."
    )


class FinalResolution(BaseModel):
    reasoning_chain: str = Field(
        description="Detailed step-by-step reasoning referencing specific policies and rules."
    )
    action_taken: ActionTaken
    customer_response: str = Field(
        description="Clear, professional, and empathetic message to the customer."
    )

In [ ]:
import json

SYSTEM_PROMPT = """You are GlobalCart's automated Single-Agent Customer Support Resolver.
Your goal is to investigate customer support tickets, execute function calls to verify data against policies, and determine the exact resolution.

Execution Protocol:
1. Always query order details first (`get_order_details`).
2. Query user profile using the user_id retrieved from the order (`get_user_profile`).
3. Evaluate eligibility using `check_return_policy`.
4. If eligible and no escalation is needed, execute `process_refund`.
5. Produce the final structured evaluation adhering strictly to policy guidelines.
"""


def run_agent(ticket_text: str) -> dict[str, any]:
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": ticket_text},
    ]

    tools_called = []

    for _ in range(6):
        response = client.chat.completions.create(
            model="gpt-5.4-mini", messages=messages, tools=tools_schema, tool_choice="auto"
        )

        msg = response.choices[0].message
        messages.append(msg)

        if not msg.tool_calls:
            break

        for tool_call in msg.tool_calls:
            func_name = tool_call.function.name
            args = json.loads(tool_call.function.arguments)

            if func_name not in tools_called:
                tools_called.append(func_name)

            result = tools_map[func_name](**args)

            messages.append(
                {"role": "tool", "tool_call_id": tool_call.id, "content": json.dumps(result)}
            )

    final_response = client.beta.chat.completions.parse(
        model="gpt-5.4-mini",
        messages=messages
        + [{"role": "system", "content": "Format the final verdict according to the schema."}],
        response_format=FinalResolution,
    )

    output = final_response.choices[0].message.parsed.model_dump()
    output["action_taken"]["tools_called"] = tools_called
    return output

In [71]:
scenarios = [
    {
        "id": "SC-01",
        "ticket": "Hi, my order ORD-1001 arrived with a broken screen. I need a refund.",
    },
    {"id": "SC-02", "ticket": "I would like to return order ORD-1002, I just changed my mind."},
    {
        "id": "SC-03",
        "ticket": "Order ORD-1003 was delivered late and damaged, please process refund.",
    },
]

results = []
for sc in scenarios:
    print(f"--- Running {sc['id']} ---")
    res = run_agent(sc["ticket"])
    results.append(
        {
            "Scenario": sc["id"],
            "Decision": res["action_taken"]["decision"],
            "Refund": res["action_taken"]["refund_amount"],
            "Tools Called": ", ".join(res["action_taken"]["tools_called"]),
            "Customer Response": res["customer_response"][:80] + "...",
        }
    )

print("\nExecution Completed Successfully.")

--- Running SC-01 ---


RateLimitError: Error code: 429 - {'error': {'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.', 'type': 'insufficient_quota', 'param': None, 'code': 'insufficient_quota'}}

In [ ]:
df_results = pd.DataFrame(results)
print(df_results.to_string(index=False))

Empty DataFrame
Columns: []
Index: []
